In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

In [ ]:
# Folder containing all the .zip files
zip_dir = Path("NEM_market_notice_CSV_Data")

# Columns you want to keep
wanted_cols = ["EFFECTIVEDATE", "EXTERNALREFERENCE"]

frames = []

In [ ]:
for zip_path in zip_dir.glob("*.zip"):
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            # Find CSV files inside the zip
            csv_files = [name for name in zf.namelist() if name.lower().endswith(".csv")]

            for csv_name in csv_files:
                with zf.open(csv_name) as f:
                    # Read only the required columns
                    df = pd.read_csv(f, usecols=wanted_cols,header=1)

                    # Convert DATETIME to pandas datetime
                    df["EFFECTIVEDATE"] = pd.to_datetime(df["EFFECTIVEDATE"])

                    frames.append(df)

    except Exception as e:
        print(f"Failed to process {zip_path.name}: {e}")

In [ ]:
# Concatenate all extracted data
final_df = pd.concat(frames, ignore_index=True)

# Sort by DATETIME
final_df = final_df.sort_values("EFFECTIVEDATE").reset_index(drop=True)

print(final_df.head())
print(final_df.shape)

In [ ]:
final_df.dropna(inplace=True)

In [ ]:
final_df_filtered = final_df[final_df["EXTERNALREFERENCE"].map(lambda x: "LOR" in x.upper())]

In [ ]:
import re

def extract_lor(text):
    pattern = r'LOR(\d{1,2})'
    match = re.search(pattern, text.upper())
    if match:
        return match.group()  # Returns just the 1-2 chars after LOR_
    return None

In [ ]:
final_df_filtered["LOR_type"] = final_df_filtered["EXTERNALREFERENCE"].map(extract_lor)

In [ ]:
final_df_filtered.dropna(subset="LOR_type",inplace=True)

In [ ]:
def extract_before_region(text):
    pattern = r'(\S+)\s+region'
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        return match.group(1)  # Returns word before "region"
    return None

In [ ]:
final_df_filtered["REGIONID"] = final_df_filtered["EXTERNALREFERENCE"].map(extract_before_region)+"1"

In [ ]:
final_df_filtered[final_df_filtered["REGIONID"]=="ALL1"]["EXTERNALREFERENCE"].tolist()

In [ ]:
items = final_df_filtered[final_df_filtered["REGIONID"].isna()]["EXTERNALREFERENCE"].tolist()[:]

In [ ]:
items

In [ ]:
for it in items:
    final_df_filtered.loc[final_df_filtered["EXTERNALREFERENCE"]==it,"REGIONID"] = "NSW1"

In [ ]:
final_df_filtered.loc[final_df_filtered["REGIONID"]=="NSW","REGIONID"] = "NSW1"

In [ ]:
final_df_filtered_LOR1 = final_df_filtered.loc[final_df_filtered["LOR_type"]=="LOR1"]

In [ ]:
final_df_filtered_LOR2 = final_df_filtered.loc[final_df_filtered["LOR_type"]=="LOR2"]

In [ ]:
final_df_filtered["REGIONID"].unique()

In [ ]:
final_df_filtered["REGIONID"] = final_df_filtered["REGIONID"].map(lambda x: x.upper().replace("TASMANIA1","TAS1").replace("WALES1","NSW").replace("THE","").replace("VICTORIA1","VIC1").replace("(","").replace("VICTORIAN1","VIC1").replace("-","").replace("QUEENSLAND1","QLD1").replace("11","1"))

In [ ]:
final_df_filtered_v2 = final_df_filtered.loc[final_df_filtered["REGIONID"]!='NAME]1']

In [ ]:
final_df_filtered_v2 = final_df_filtered_v2.loc[final_df_filtered["REGIONID"]!='XX1']

In [ ]:
final_df_filtered_v2["REGIONID"].unique()

In [ ]:
# Your replacement list (length = 5)
replacement_list = ['SA1', 'QLD1', 'TAS1', 'NSW1', 'VIC1']

In [ ]:
# Split dataframe
df_all = final_df_filtered_v2[final_df_filtered_v2["REGIONID"] == 'ALL1']
df_other = final_df_filtered_v2[final_df_filtered_v2["REGIONID"] != 'ALL1']

In [ ]:
df_all

In [ ]:
# Repeat rows and replace
df_all_expanded = (
    df_all.loc[df_all.index.repeat(len(replacement_list))]
    .assign(**{
        "REGIONID": replacement_list * len(df_all)
    })
)

In [ ]:
df_all_expanded

In [ ]:
# Combine back
result = pd.concat([df_other, df_all_expanded], ignore_index=True)

In [ ]:
result["REGIONID"].unique()

In [ ]:
result.to_csv("semi_cleaned_regionized_market_notice.csv",index=False)

# For the Actual data

In [ ]:
df = pd.read_csv("semi_cleaned_regionized_market_notice.csv")

In [ ]:
Actual = df.loc[df["EXTERNALREFERENCE"].map(lambda x: "ACTUAL" in x.upper())]

In [ ]:
Actual["EXTERNALREFERENCE"].unique()

In [ ]:
Actual['EVENT'] = Actual['EXTERNALREFERENCE'].str.lower().apply(
    lambda x: 'end' if 'cancel' in x else 'start'
)

In [ ]:
Actual

In [ ]:
df_notices = Actual.sort_values(['REGIONID', 'LOR_type', 'EFFECTIVEDATE'])

In [ ]:
intervals = []

for (region, lor), group in df_notices.groupby(['REGIONID', 'LOR_type']):
    start_time = None

    for _, row in group.iterrows():
        if row['EVENT'] == 'start':
            start_time = row['EFFECTIVEDATE']

        elif row['EVENT'] == 'end' and start_time is not None:
            intervals.append({
                'REGIONID': region,
                'LOR_type': lor,
                'start': start_time,
                'end': row['EFFECTIVEDATE']
            })
            start_time = None

In [ ]:
df_intervals = pd.DataFrame(intervals)

In [ ]:
df_intervals.to_csv("Actual_intervals.csv",index=False)

In [ ]:
import pandas as pd
from datetime import datetime

# Define date range
start_date = '2015-01-01 00:00:00'
end_date = '2024-07-30 23:55:00'  # Ends at last 5-min interval of Sep 30

# Create 5-minute frequency datetime index
date_range = pd.date_range(
    start=start_date, 
    end=end_date, 
    freq='5min'
)

# Create DataFrame with single date column
df_ts = pd.DataFrame({'date': date_range})

print(f"DataFrame shape: {df_ts.shape}")
print(df_ts.head())
print(df_ts.tail())

In [ ]:
for lor in ['LOR1', 'LOR2', 'LOR3']:
    df_ts[f'ACTUAL_{lor}'] = 0

In [ ]:
df_ts["REGIONID"] = 0

In [ ]:
df_ts_expanded = (
    df_ts.loc[df_ts.index.repeat(len(df_intervals['REGIONID'].unique().tolist()))]
    .assign(**{
        "REGIONID": df_intervals['REGIONID'].unique().tolist() * len(df_ts)
    })
)

In [ ]:
df_ts_expanded

In [ ]:
for _, row in df_intervals.iterrows():
    mask = (
        (df_ts_expanded['REGIONID'] == row['REGIONID']) &
        (df_ts_expanded['date'] >= row['start']) &
        (df_ts_expanded['date'] <= row['end'])
    )
    
    df_ts_expanded.loc[mask, f"ACTUAL_{row['LOR_type']}"] = 1

In [ ]:
df_ts_expanded.drop(columns=["LOR1_flag","LOR2_flag","LOR3_flag"],inplace=True)

In [ ]:
df_ts_expanded[df_ts_expanded["ACTUAL_LOR1"]==1]

In [ ]:
df_ts_expanded.to_csv("Actual_LOR_Final.csv",index=False)

# For the Forecast data

In [ ]:
df = pd.read_csv("semi_cleaned_regionized_market_notice.csv",parse_dates=["EFFECTIVEDATE"])

In [ ]:
df.head()

In [ ]:
Forecast = df.loc[df["EXTERNALREFERENCE"].map(lambda x: "FORECAST" in x.upper())]

In [ ]:
Forecast['EVENT'] = Forecast['EXTERNALREFERENCE'].str.lower().apply(
    lambda x: 'end' if 'cancel' in x else 'start'
)

In [ ]:
df_notices = Forecast.sort_values(['REGIONID', 'LOR_type', 'EFFECTIVEDATE'])

In [ ]:
intervals = []

for (region, lor), group in df_notices.groupby(['REGIONID', 'LOR_type']):
    start_time = None

    for _, row in group.iterrows():
        if row['EVENT'] == 'start':
            start_time = row['EFFECTIVEDATE']

        elif row['EVENT'] == 'end' and start_time is not None:
            intervals.append({
                'REGIONID': region,
                'LOR_type': lor,
                'start': start_time,
                'end': row['EFFECTIVEDATE']
            })
            start_time = None

In [ ]:
df_intervals = pd.DataFrame(intervals)

In [ ]:
df_intervals

In [ ]:
df_intervals.to_csv("Forecast_intervals.csv",index=False)

In [ ]:
import pandas as pd
from datetime import datetime

# Define date range
start_date = '2015-01-01 00:00:00'
end_date = '2024-07-30 23:55:00'  # Ends at last 5-min interval of Sep 30

# Create 5-minute frequency datetime index
date_range = pd.date_range(
    start=start_date, 
    end=end_date, 
    freq='5min'
)

# Create DataFrame with single date column
df_ts = pd.DataFrame({'date': date_range})

print(f"DataFrame shape: {df_ts.shape}")
print(df_ts.head())
print(df_ts.tail())

In [ ]:
for lor in ['LOR1', 'LOR2', 'LOR3']:
    df_ts[f'FORECAST_{lor}'] = 0

In [ ]:
df_ts["REGIONID"] = 0

In [ ]:
df_ts_expanded = (
    df_ts.loc[df_ts.index.repeat(len(df_intervals['REGIONID'].unique().tolist()))]
    .assign(**{
        "REGIONID": df_intervals['REGIONID'].unique().tolist() * len(df_ts)
    })
)

In [ ]:
for _, row in df_intervals.iterrows():
    mask = (
        (df_ts_expanded['REGIONID'] == row['REGIONID']) &
        (df_ts_expanded['date'] >= row['start']) &
        (df_ts_expanded['date'] <= row['end'])
    )
    
    df_ts_expanded.loc[mask, f"FORECAST_{row['LOR_type']}"] = 1

In [ ]:
df_ts_expanded

In [ ]:
df_ts_expanded[df_ts_expanded["FORECAST_LOR1"]==1]

In [ ]:
df_ts_expanded.to_csv("FORECAST_LOR_Final.csv",index=False)